# Notebook 11: Ensembles and Stacking

## Why This Matters

Ensembles consistently win machine learning competitions (Kaggle winners routinely use stacking) and improve production model performance. The theory is well-understood: by combining diverse models, you reduce variance without significantly increasing bias. Understanding *why* ensembles work -- and when they don't -- separates practitioners from those who blindly average predictions.

## Background

**Bias-variance tradeoff**: Total error = bias^2 + variance + irreducible noise. Bagging reduces variance; boosting reduces bias. Random forests reduce variance more than single bagging because random feature selection decorrelates trees.

**Voting**: Hard voting uses majority class; soft voting averages probabilities (usually better if models are calibrated).

**Bagging**: Bootstrap aggregation trains each model on a different bootstrap sample. Out-of-bag (OOB) samples serve as a free validation set.

**Stacking**: Train level-0 learners and use their out-of-fold (OOF) predictions as features for a level-1 meta-learner. OOF prevents leakage that would occur if base learners saw their own training data.

**Blending**: Simpler version of stacking -- hold out a fixed validation set for training the meta-learner. Faster but wastes data.

## Community Context

Kaggle competitions demonstrated that 2-3 layer stacking with diverse base models (linear, tree-based, neural) and a linear meta-learner is the gold standard for tabular data. The key insight: diversity among base models matters more than individual model performance. A weak model that is uncorrelated with other models adds more value than a strong but correlated one.

In [ ]:
%pip install -q scikit-learn xgboost numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, BaggingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

np.random.seed(42)

X, y = make_classification(
    n_samples=3000, n_features=20, n_informative=10,
    n_redundant=5, n_clusters_per_class=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Bias-variance decomposition via repeated train/test splits
def bias_variance_decomp(ModelClass, n_bootstrap=50, **kwargs):
    preds = []
    for seed in range(n_bootstrap):
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=seed
        )
        m = ModelClass(random_state=seed, **kwargs)
        m.fit(Xtr, ytr)
        preds.append(m.predict(X_test))
    preds = np.array(preds)  # (n_bootstrap, n_test)
    mean_pred = preds.mean(axis=0)
    bias_sq = ((mean_pred - y_test) ** 2).mean()
    variance = preds.var(axis=0).mean()
    return bias_sq, variance

print("\nBias-Variance Decomposition (50 bootstrap rounds):")
print(f"{"Model":30s} {"Bias^2":>10s} {"Variance":>10s} {"B+V":>10s}")
print("-" * 65)
for name, cls, kw in [
    ("Decision Tree (depth=None)", DecisionTreeClassifier, {}),
    ("Decision Tree (depth=3)", DecisionTreeClassifier, {"max_depth": 3}),
    ("Random Forest (100)", RandomForestClassifier, {"n_estimators": 100}),
]:
    b, v = bias_variance_decomp(cls, **kw)
    print(f"{name:30s} {b:>10.4f} {v:>10.4f} {b+v:>10.4f}")

## 2. Voting Classifiers

Hard voting counts majority class predictions. Soft voting averages predicted probabilities. Soft voting is almost always better when base models produce calibrated probabilities, because it uses richer information than just the argmax prediction.

In [ ]:
# Hard and soft voting ensembles
lr = LogisticRegression(max_iter=1000, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gnb = GaussianNB()

hard_vote = VotingClassifier(
    estimators=[("lr", lr), ("rf", rf), ("gnb", gnb)],
    voting="hard"
)
soft_vote = VotingClassifier(
    estimators=[("lr", lr), ("rf", rf), ("gnb", gnb)],
    voting="soft"
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_to_eval = [
    ("Logistic Regression", lr),
    ("Random Forest", rf),
    ("GradientBoosting", gb),
    ("Naive Bayes", gnb),
    ("Hard Voting", hard_vote),
    ("Soft Voting", soft_vote),
]

print("5-fold CV AUC:")
for name, model in models_to_eval:
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"  {name:25s}  {scores.mean():.4f} +/- {scores.std():.4f}")

## 3. BaggingClassifier and Variance Reduction

Bagging trains each estimator on a bootstrap sample (with replacement) of the training data. As ensemble size grows, variance decreases toward the irreducible variance of a single model. Random forests add feature subsampling, further decorrelating trees and improving over plain bagging.

In [ ]:
# BaggingClassifier -- bootstrap aggregation of any base estimator
base_tree = DecisionTreeClassifier(max_depth=8, random_state=42)

bagging_models = {
    "Single Deep Tree": DecisionTreeClassifier(max_depth=None, random_state=42),
    "Bagging (10 trees)": BaggingClassifier(base_tree, n_estimators=10, random_state=42),
    "Bagging (50 trees)": BaggingClassifier(base_tree, n_estimators=50, random_state=42),
    "Bagging (100 trees)": BaggingClassifier(base_tree, n_estimators=100, random_state=42),
    "Random Forest (100)": RandomForestClassifier(n_estimators=100, random_state=42),
}

print("Effect of ensemble size on variance:")
print(f"{"Model":30s} {"CV AUC":>10s} {"Std":>8s}")
print("-" * 55)
for name, model in bagging_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name:30s} {scores.mean():.4f}   {scores.std():.4f}")

## 4. Stacking with OOF Predictions

The critical detail in stacking: base learners' predictions used to train the meta-learner must be out-of-fold predictions. If a base learner sees its own training samples when generating meta-features, the meta-learner will over-weight it because its predictions are unrealistically good. OOF stacking is equivalent to cross-validation at the meta-feature level.

In [ ]:
# Stacking with out-of-fold (OOF) predictions
# Level-0: diverse base learners; Level-1: meta-learner
level0_estimators = [
    ("lr", LogisticRegression(max_iter=1000, C=0.1)),
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)),
    ("gnb", GaussianNB()),
]
meta_learner = LogisticRegression(max_iter=1000)

stack = StackingClassifier(
    estimators=level0_estimators,
    final_estimator=meta_learner,
    cv=5,
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1
)

# Manual OOF stacking for inspection
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_train), len(level0_estimators)))

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    Xf_tr, Xf_val = X_train[tr_idx], X_train[val_idx]
    yf_tr = y_train[tr_idx]
    for model_idx, (name, model) in enumerate(level0_estimators):
        from sklearn.base import clone
        m = clone(model)
        m.fit(Xf_tr, yf_tr)
        oof_preds[val_idx, model_idx] = m.predict_proba(Xf_val)[:, 1]

print("OOF base learner AUCs:")
for i, (name, _) in enumerate(level0_estimators):
    auc = roc_auc_score(y_train, oof_preds[:, i])
    print(f"  {name}: {auc:.4f}")

# Train meta-learner on OOF
meta = LogisticRegression(max_iter=1000)
meta.fit(oof_preds, y_train)

# Re-train base learners on full train, predict test
test_preds = np.zeros((len(X_test), len(level0_estimators)))
for model_idx, (name, model) in enumerate(level0_estimators):
    from sklearn.base import clone
    m = clone(model)
    m.fit(X_train, y_train)
    test_preds[:, model_idx] = m.predict_proba(X_test)[:, 1]

meta_probs = meta.predict_proba(test_preds)[:, 1]
meta_auc = roc_auc_score(y_test, meta_probs)
print(f"\nManual stacking AUC on test: {meta_auc:.4f}")

# Compare with sklearn StackingClassifier
stack_scores = cross_val_score(stack, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=-1)
print(f"sklearn StackingClassifier CV AUC: {stack_scores.mean():.4f} +/- {stack_scores.std():.4f}")

## 5. Blending vs Stacking

Blending is simpler: hold out 20% of training data as a "blend set". Train base models on the 80%, generate blend-set predictions, train the meta-learner on those predictions, then combine. Blending is faster and easier to implement but wastes 20% of training data. For small datasets, OOF stacking is clearly better.

In [ ]:
# Blending vs Stacking
# Blending: hold out a fixed "blend set" instead of using OOF
X_tr2, X_blend, y_tr2, y_blend = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

blend_preds_val = np.zeros((len(X_blend), len(level0_estimators)))
blend_preds_test = np.zeros((len(X_test), len(level0_estimators)))

from sklearn.base import clone
for model_idx, (name, model) in enumerate(level0_estimators):
    m = clone(model)
    m.fit(X_tr2, y_tr2)
    blend_preds_val[:, model_idx] = m.predict_proba(X_blend)[:, 1]
    blend_preds_test[:, model_idx] = m.predict_proba(X_test)[:, 1]

blend_meta = LogisticRegression(max_iter=1000)
blend_meta.fit(blend_preds_val, y_blend)
blend_test_probs = blend_meta.predict_proba(blend_preds_test)[:, 1]
blend_auc = roc_auc_score(y_test, blend_test_probs)

# Simple average ensemble
avg_probs = test_preds.mean(axis=1)
avg_auc = roc_auc_score(y_test, avg_probs)

print("Final comparison on test set:")
print(f"  Simple average:  AUC = {avg_auc:.4f}")
print(f"  Blending:        AUC = {blend_auc:.4f}")
print(f"  Stacking (OOF):  AUC = {meta_auc:.4f}")

# Show meta-learner coefficients (how much it weights each base model)
print("\nMeta-learner weights (stacking):")
for i, (name, _) in enumerate(level0_estimators):
    print(f"  {name}: {meta.coef_[0][i]:.4f}")

## Summary

| Method | Key Mechanism | Reduces | Best For |
|--------|--------------|---------|----------|
| Voting (hard) | Majority class | Variance | Fast baselines |
| Voting (soft) | Average probabilities | Variance | Calibrated models |
| Bagging | Bootstrap sampling | Variance | High-variance base models |
| Random Forest | Bagging + feature subsampling | Variance | General tabular ML |
| Stacking (OOF) | Meta-learner on OOF | Both | Kaggle, production |
| Blending | Meta-learner on holdout | Both | Simplicity, speed |

**Interview tip:** Diversity is the key ingredient in ensembles. Averaging two identical models gives no benefit. The best ensembles combine models with different inductive biases: linear (captures global patterns), trees (captures local patterns), neural networks (captures complex interactions). When base model predictions are highly correlated, blending them produces diminishing returns.